# <b>PS-S3E1 🎊 | 📊 EDA → ⚙️FE → CatBoost 🐈</b>

![PlaygroundSeries Season 3, Episode 1](https://storage.googleapis.com/kaggle-competitions/kaggle/44629/logos/header.png?t=2022-12-21-20-35-21)

### <span style='color:#a600ff'>Happy New Year!</span> 🎉
May this year bring good health, happiness and success at whatever you endeavor!

___

# <a id="Agenda"><b>📝 Agenda</b></a>
>1. [📚 Loading libraries and files](#Loading)
>2. [🔍 Exploratory Data Analysis](#EDA)
>3. [⚙️ Feature Engineering](#FeatureEngineering)
>4. [✅ Cross-validation Method](#Validation)
>5. [🏋️ Model Training & Inference](#Training)
>6. [📮 Post-processing & Submission](#Submit)

___
# <a id="Loading"><b>1 <span style='color:#a600ff'>•</span> Loading libraries and files 📚</b></a>

In [ ]:
%%capture

# Intel® Extension for Scikit-learn installation:
!pip install scikit-learn-intelex

import os
import warnings

import numpy as np
import pandas as pd

import seaborn as sns
sns.set_style('whitegrid')

import matplotlib.pyplot as plt

%matplotlib inline

from pathlib import Path
from tqdm.notebook import tqdm

from typing import Optional, Type, List, Union, Dict, Tuple
from dataclasses import dataclass, field

from sklearnex import patch_sklearn
patch_sklearn()

# Mute warnings
warnings.filterwarnings('ignore')

In [ ]:
!tree ../input/

### <span style='color:#a600ff'>Loading data</span>

In [ ]:
# Set the path to the directory containing the input data files
data_dir = Path('../input/playground-series-s3e1')

# Load the training data into a Pandas DataFrame
df_train = pd.read_csv(data_dir / 'train.csv', index_col='id')

# Load the test data into a Pandas DataFrame
df_test = pd.read_csv(data_dir / 'test.csv', index_col='id')

In [ ]:
# Target column
TARGET = df_train.columns.difference(df_test.columns)[0]

# Features (predictive attributes)
features = df_train.columns[df_train.columns != TARGET]

# Print list of features and target column
print(f'Features: {features.tolist()}\n') 
print(f'Target: {TARGET}')

### <span style='color:#a600ff'>Loading original data</span>
📌 This part has been updated and largely inspired by this notebook:
> [S03E01: Boost with Original data 🔥](https://www.kaggle.com/code/phongnguyen1/s03e01-boost-with-original-data?scriptVersionId=115366781) by [Phong Nguyen](https://www.kaggle.com/phongnguyen1)

In [ ]:
# Import original California Housing dataset
from sklearn.datasets import fetch_california_housing

# Load dataset
original = fetch_california_housing()

# Check feature and target names
assert original['feature_names'] == list(features)
assert original['target_names'][0] == TARGET

# Create DataFrame from original data and feature names
df_original = pd.DataFrame(original['data'], columns=features)

# Add target column to DataFrame
df_original[TARGET] = original['target']

In [ ]:
print(original.DESCR)

⬆️ [Back to the top](#Agenda) ⬆️
___
# <a id="EDA"><b>2 <span style='color:#a600ff'>•</span> Exploratory Data Analysis 🔍</b></a>

📌 This part has been updated and largely inspired by this website:
> [The California housing dataset](https://inria.github.io/scikit-learn-mooc/python_scripts/datasets_california_housing.html) by scikit-learn developers.

In [ ]:
df_train.head()

In [ ]:
df_train.info()

**Insights:**
* the dataset contains 37,137 samples and 8 features;
* all features are numerical features encoded as floating number;
* there is no missing values.

In [ ]:
print('Train set - dimensions:\t', df_train.shape)
print('Test set  - dimensions:\t', df_test.shape)

### <span style='color:#a600ff'>Univariate analysis</span>

#### Feature analysis:

In [ ]:
# Adding the original dataset to the train set
df_combined = pd.concat([df_train, df_original]).reset_index(drop=True)

# Set the number of bins for the histograms
bins = 50

# Set the number of rows and columns for the subplots
nrows = 4
ncols = 2

# Create the figure and subplots
fig, axs = plt.subplots(nrows, ncols, figsize=(18, 20))

# Flatten the array of subplots (2D array) into a 1D array
axs = axs.flatten()

# Set the color palette
palette = sns.color_palette('plasma')

# Loop through each column and plot the histograms on the corresponding subplot
for i, feature in enumerate(features):
    # Plot the histograms
    sns.histplot(df_combined[feature], bins=bins, ax=axs[i], alpha=0.8, color=palette[0], edgecolor='black', label='Combined data')
    sns.histplot(df_train[feature], bins=bins, ax=axs[i], alpha=0.8, color=palette[4], edgecolor='black', label='Synthetic data')
    sns.histplot(df_original[feature], bins=bins, ax=axs[i], alpha=0.8, color=palette[5], edgecolor='black', label='Original data')
    
    # Add a legend and labels
    axs[i].legend(loc='upper right', fontsize=16)
    axs[i].set_xlabel(feature, fontsize=16)
    axs[i].set_ylabel('')
    
    # Set the font size of the tick labels
    axs[i].tick_params(axis='both', labelsize=16)
    
    # Remove the grid lines at y=0
    axs[i].set_yticks(axs[i].get_yticks()[1:])
    
    
# Tighten the layout and show the figure
plt.tight_layout()
plt.show()

#### Target analysis:

In [ ]:
# Create a figure
fig, ax = plt.subplots(figsize=(10, 6))

# Plot the histograms
sns.histplot(data=df_combined[TARGET], ax=ax, alpha=0.8, color=palette[0], edgecolor='black', label='Combined data')
sns.histplot(data=df_train[TARGET], ax=ax, alpha=0.8, color=palette[4], edgecolor='black', label='Synthetic data')
sns.histplot(data=df_original[TARGET], ax=ax, alpha=0.8, color=palette[5], edgecolor='black', label='Original data')

# Add a legend and labels
ax.legend(loc='upper left', fontsize=14)
ax.set_xlabel(TARGET, fontsize=16)
ax.set_ylabel('')

# Set the font size of the tick labels
ax.tick_params(axis='both', labelsize=16)

# Remove the grid lines at y=0
ax.set_yticks(ax.get_yticks()[1:])

# Tighten the layout and show the figure
plt.tight_layout()

plt.show()

In [ ]:
# Maximum value of the target
target_max_val = df_combined[TARGET].max()
f'{TARGET} max value: {target_max_val}'

**Insights:**
* The median income `MedInc` is a right skewed distribution with a long tail. Meaning that the salary of people is more or less normally distributed but there is some people getting a high salary.
* Regarding the average house age `HouseAge`, the distribution is more or less uniform.
* The target `MedHouseVal` distribution has a long tail as well. In addition, we have a threshold-effect for high-valued houses: all houses with a price above 5 are given the value 5.00001.
* Focusing on the average rooms `AveRooms`, average bedrooms `AveBedrms`, average occupation `AveOccup`, and population `Population`, the range of the data is large with unnoticeable bin for the largest values. It means that there are very high and few values, possibly outlying values. 

Moreover, the distributions are quite close, so that it may be relevant to combine both original and synthetically-generated data. From now on, we will deal with combined data.

In [ ]:
df_combined.head()

In [ ]:
df_combined.info()

**Insights:**
* the dataset contains 57,777 samples and 8 features;
* all features are numerical features encoded as floating number;
* there is no missing values.

Remember that there are very high and few values, possibly outlying values in `AveRooms`, `AveBedrms`, `AveOccup`, and `Population`.

We can have a closer look to these features:

In [ ]:
# Get summary statistics for the selected columns
features_of_interest = ['AveRooms', 'AveBedrms', 'AveOccup', 'Population']
df_combined[features_of_interest].describe()

**Insight:** Comparing the maximum `max` and 75th percentile `75%` values for each feature reveals a significant difference, indicating the presence of extreme values or outliers. This aligns with the observation that there are relatively few data points with large values for the features related to average number of rooms, bedrooms, occupation, and population.

In [ ]:
# Random subsampling
rng = np.random.RandomState(0)
indices = rng.choice(np.arange(df_combined.shape[0]), size=500,replace=False)

# Drop the unwanted columns
cols_to_drop = ['Longitude', 'Latitude']
subset = df_combined.iloc[indices].drop(columns=cols_to_drop)

# Quantize the target and keep the midpoint for each interval
subset[TARGET] = pd.qcut(subset[TARGET], 6, retbins=False)
subset[TARGET] = subset[TARGET].apply(lambda x: x.mid)

We can make a pair plot of all features and the target but dropping the longitude and latitude. We will quantize the target such that we can create proper histogram.

In [ ]:
_ = sns.pairplot(data=subset, hue=TARGET, palette='plasma')

**Insights:**
* confirmation that some features have extreme values, possibly outliers.
* the median income `MedInc` is helpful to distinguish high-valued from low-valued houses.

⬆️ [Back to the top](#Agenda) ⬆️
___
# <a id="FeatureEngineering"><b>3 <span style='color:#a600ff'>•</span> Feature Engineering ⚙️</b></a>

📌 This part has been updated and largely inspired by theses notebooks and discussions:
> * [PS S3E1 | Coordinates - key to victory](https://www.kaggle.com/code/dmitryuarov/ps-s3e1-coordinates-key-to-victory) by [Dmitry Uarov](https://www.kaggle.com/dmitryuarov)
> * [Lat / Long Feature Engineering tricks from previous competitions](https://www.kaggle.com/competitions/playground-series-s3e1/discussion/376210) by [The Devastator](https://www.kaggle.com/thedevastator)
> * [Simple feature that boost your score +0.002](https://www.kaggle.com/competitions/playground-series-s3e1/discussion/376043) by [Sinan Calisir](https://www.kaggle.com/snnclsr)
> * [Coordinates using google maps](https://www.kaggle.com/competitions/playground-series-s3e1/discussion/376542) by [Samuel Cortinhas](https://www.kaggle.com/samuelcortinhas)
> * [Coastline distance (Implementation)](https://www.kaggle.com/competitions/playground-series-s3e1/discussion/376683) by [Sergey Saharovskiy](https://www.kaggle.com/sergiosaharovskiy)

In [ ]:
df_original['is_generated'] = 0
df_train['is_generated']    = 1
df_test['is_generated']     = 1

df_combined = pd.concat([df_train, df_original], ignore_index=True)

In [ ]:
# Highlight newly created columns
df_combined[['is_generated']]

In [ ]:
# def embed_lat_long(df: pd.DataFrame, num_embedding_dimensions: int, precision: float) -> np.ndarray:
#     """
#     Embed the latitude and longitude coordinates in the given dataframe into a lower-dimensional space.
    
#     Parameters
#     ----------
#         df : pd.DataFrame
#             A dataframe with 'Latitude' and 'Longitude' columns.
#         num_embedding_dimensions : int 
#             The number of dimensions to use in the embedding.
#         precision : float
#             A scaling factor to use in the embedding.
    
#     Returns
#     -------
#         latlong : np.ndarray
#             An array of shape (num_examples, 2 * num_embedding_dimensions) containing the embedded coordinates.
#     """
#     latlong = np.expand_dims(df[['Latitude', 'Longitude']].values, axis=-1)
    
#     scaling_factor = np.exp(np.log(precision) / num_embedding_dimensions) 
#     angle_frequencies = scaling_factor ** np.arange(num_embedding_dimensions) 
#     angle_frequencies = angle_frequencies.reshape(1, 1, num_embedding_dimensions) 
    
#     latlong = latlong * angle_frequencies
#     latlong[..., ::2] = np.cos(latlong[..., ::2])
#     latlong[..., 1::2] = np.sin(latlong[..., 1::2])
#     latlong = latlong.reshape(-1, 2 * num_embedding_dimensions)
    
#     return latlong

# emb_size = 20
# precision = 1e6 

# df_combined['exp_latlong1'] = [lat[0] for lat in embed_lat_long(df_combined, emb_size, precision)]
# df_combined['exp_latlong2'] = [lat[1] for lat in embed_lat_long(df_combined, emb_size, precision)]

# df_test['exp_latlong1'] = [lat[0] for lat in embed_lat_long(df_test, emb_size, precision)]
# df_test['exp_latlong2'] = [lat[1] for lat in embed_lat_long(df_test, emb_size, precision)]

In [ ]:
# # Highlight newly created columns
# df_combined[['exp_latlong1', 'exp_latlong2']]

In [ ]:
def polar_coordinates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert the latitude and longitude values in a DataFrame to polar coordinates
    and add them as new columns in the DataFrame.
    
    Parameters
    ----------
        df : pd.DataFrame
            Input DataFrame containing 'Latitude' and 'Longitude' columns.
        
    Returns
    -------
        df : pd.DataFrame
            Output DataFrame with additional columns for the polar coordinates.
    """
    # Check that the input DataFrame has the necessary columns
    if not all(col in df.columns for col in ['Latitude', 'Longitude']):
        raise ValueError("Input DataFrame must have 'Latitude' and 'Longitude' columns.")
    
    # Convert the latitude and longitude values to polar coordinates
    df['r'] = np.sqrt(df['Latitude']**2 + df['Longitude']**2)
    df['theta'] = np.arctan2(df['Latitude'], df['Longitude'])
    
    return df

df_combined = polar_coordinates(df_combined)
df_test = polar_coordinates(df_test)

In [ ]:
# Highlight newly created columns
df_combined[['r', 'theta']]

In [ ]:
def cartesian_coordinates(df: pd.DataFrame, angles: Union[int, List[int]]) -> pd.DataFrame:
    """
    Convert the latitude and longitude values in a DataFrame to rotated
    Cartesian coordinates and add them as new columns in the DataFrame.
    
    Parameters
    ----------
        df : pd.DataFrame
            Input DataFrame containing 'Latitude' and 'Longitude' columns.
        angles : int or list of int
            Rotation angles in degrees.
        
    Returns
    -------
        df : pd.DataFrame
            Output DataFrame with additional columns for the rotated Cartesian
            coordinates.
    """
    # Check that the input DataFrame has the necessary columns
    if not all(col in df.columns for col in ['Latitude', 'Longitude']):
        raise ValueError("Input DataFrame must have 'Latitude' and 'Longitude' columns.")
    
    # Convert angles to a list if it's not already one
    if not isinstance(angles, list):
        angles = [angles]
    
    # Loop over the angles and compute the rotated coordinates
    for angle in angles:
        # Check that the rotation angle is a valid value
        if not (0 <= angle < 360):
            raise ValueError("Rotation angle must be between 0 and 360 degrees (inclusive).")
        
        # [x cos(w) - y sin(w), x sin(w) + y cos(w)]
        df['rot_' + str(angle) + '_x'] = (df['Latitude'] * np.cos(np.radians(angle))) - \
                                         (df['Longitude'] * np.sin(np.radians(angle)))
        df['rot_' + str(angle) + '_y'] = (df['Latitude'] * np.sin(np.radians(angle))) + \
                                         (df['Longitude'] * np.cos(np.radians(angle)))
    
    return df

# List of angles
angles = [15, 30, 45]

# Conversion for both train and test sets
df_combined = cartesian_coordinates(df_combined, angles)
df_test = cartesian_coordinates(df_test, angles)

In [ ]:
# Highlight newly created columns
df_combined.iloc[:, -6:]

In [ ]:
# from umap import UMAP

# def add_umap_coordinates(df: pd.DataFrame, coordinates: List[Tuple[float, float]]) -> pd.DataFrame:
#     """
#     Add UMAP-transformed coordinates as separate columns to a dataframe.

#     Parameters
#     ----------
#         df: pandas.DataFrame
#             Dataframe to which the transformed coordinates will be added.
#         coordinates: list of tuples
#             List of coordinates to be transformed, where each tuple represents a pair of (latitude, longitude) values.

#     Returns
#     -------
#         df: pandas.DataFrame
#             The input dataframe with the two additional columns 'umap_lat' and 'umap_lon' added.
#     """
#     # Fit the UMAP model to the coordinates data
#     umap = UMAP(n_components=2, n_neighbors=50, random_state=42).fit(coordinates)

#     # Use the model to transform the coordinates data
#     umap_coordinates = umap.transform(coordinates)

#     # Add the transformed coordinates to the dataframe as separate columns
#     df['umap_lat']  = umap_coordinates[:,0]
#     df['umap_long'] = umap_coordinates[:,1]

#     return df

# df_combined = add_umap_coordinates(df_combined, df_combined[['Latitude', 'Longitude']].values)
# df_test = add_umap_coordinates(df_test, df_test[['Latitude', 'Longitude']].values)

In [ ]:
# # Highlight newly created columns
# df_combined[['umap_lat', 'umap_long']]

In [ ]:
%%capture
!pip install reverse_geocoder

In [ ]:
import reverse_geocoder as rg
from sklearn.preprocessing import LabelEncoder

def geocoder(df: pd.DataFrame) -> List[Dict[str, Union[str, float]]]:
    """
    Use the reverse geocoder library to get the place names for the
    latitude and longitude values in the input DataFrame.
    
    Parameters
    ----------
        df : pd.DataFrame
            Input DataFrame containing 'Latitude' and 'Longitude' columns.
        
    Returns
    -------
        results : list of dict
            List of dictionaries containing place names and other information
            for each set of coordinates.
    """   
    coordinates = list(zip(df['Latitude'], df['Longitude']))
    results = rg.search(coordinates)
    
    return results

def process_places(df: pd.DataFrame, places: List[str]) -> pd.DataFrame:
    """
    Process the place names in the input DataFrame and encode them as integers.
    
    Parameters
    ----------
        df : pd.DataFrame
            Input DataFrame containing a 'place' column.
        places : list of str
            List of place names to keep as separate categories.
        
    Returns
    -------
        df : pd.DataFrame
            Output DataFrame with the 'place' column encoded as integers.
    """
    # Replace place names that are not in the list of places with 'Other'
    df['place'] = df['place'].apply(lambda x: x if x in places else 'Other')
    
    # Encode the place names as integers
    le = LabelEncoder()
    df['place'] = le.fit_transform(df['place'])
    
    return df

# Get the place names for the latitude and longitude values in the DataFrames
results = geocoder(df_combined)
df_combined['place'] = [x['admin2'] for x in results]

results = geocoder(df_test)
df_test['place'] = [x['admin2'] for x in results]

# Define the list of places to keep as separate categories
places = ['Los Angeles County', 'Orange County', 'Kern County',
          'Alameda County', 'San Francisco County', 'Ventura County',
          'Santa Clara County', 'Fresno County', 'Santa Barbara County',
          'Contra Costa County', 'Yolo County', 'Monterey County',
          'Riverside County', 'Napa County']

# Process the place names in the DataFrames
df_combined = process_places(df_combined, places)
df_test = process_places(df_test, places)

In [ ]:
# Highlight newly created columns
df_combined[['place']]

In [ ]:
from sklearn.decomposition import PCA

def pca_coordinates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Use principal component analysis (PCA) to extract the main components
    of the latitude and longitude values in the input DataFrame and add
    them as new columns in the DataFrame.
    
    Parameters
    ----------
        df : pd.DataFrame
            Input DataFrame containing 'Latitude' and 'Longitude' columns.
        
    Returns
    -------
        df : pd.DataFrame
            Output DataFrame with additional 'pca_lat' and 'pca_long' columns
            for the main components of the latitude and longitude values.
    """
    # Fit a PCA model to the latitude and longitude values
    coordinates = df[['Latitude', 'Longitude']].values
    pca = PCA().fit(coordinates)
    
    # Transform the coordinates using the PCA model and add the main components as new columns
    df['pca_lat'] = pca.transform(df[['Latitude', 'Longitude']])[:, 0]
    df['pca_long'] = pca.transform(df[['Latitude', 'Longitude']])[:, 1]
    
    return df

# Extract the main components of the latitude and longitude values in the DataFrames
df_combined = pca_coordinates(df_combined)
df_test = pca_coordinates(df_test)

In [ ]:
# Highlight newly created columns
df_combined[['pca_lat', 'pca_long']]

In [ ]:
# from geopy.distance import geodesic

# @dataclass
# class DistanceParams:
#     """
#     A dataclass to hold the parameters for the `compute_distances` function.
    
#     Attributes
#     ----------
#         df : pandas.DataFrame
#             The DataFrame to compute the distances for.
#         lat_col : str
#             The name of the column in df containing the latitude values.
#         lon_col : str
#             The name of the column in df containing the longitude values.
#         cities : dict
#             A dictionary mapping city names to (latitude, longitude) pairs.
#         unit (optional) : str
#             The unit of distance to use. Acceptable values are 'km', 'miles', 'ft', 'meters', 'yards', and 'nautical'. Default is 'km'.
#         new_cities (optional) : dict
#             A dictionary of additional cities to include in the calculation. Default is an empty dictionary.
#     """
#     df: pd.DataFrame
#     lat_col: str
#     lon_col: str
#     cities: Dict[str, Tuple[float, float]]
#     unit: str = 'km'
#     new_cities: Dict[str, Tuple[float, float]] = field(default_factory=dict)
#     # Use field, since mutable objects are not allowed as default values for fields in a dataclass

# def compute_distances(params: DistanceParams) -> pd.DataFrame:
#     """
#     Compute the distances from each row in the given DataFrame to a set of cities and add the results as new columns.
    
#     Parameters
#     ----------
#         params : DistanceParams)
#             An instance of the DistanceParams dataclass containing the input parameters for the function.
    
#     Returns
#     -------
#         df : pandas.DataFrame
#             A copy of params.df with additional columns for the distance to each city and the nearest city.
#     """
#     df = params.df
#     lat_col = params.lat_col
#     lon_col = params.lon_col
#     cities = params.cities
#     unit = params.unit
#     new_cities = params.new_cities
    
#     # Handle cases where lat_col or lon_col do not exist in df
#     if lat_col not in df.columns:
#         raise ValueError(f"Column '{lat_col}' does not exist in DataFrame.")
#     if lon_col not in df.columns:
#         raise ValueError(f"Column '{lon_col}' does not exist in DataFrame.")
    
#     # Handle cases where cities is not a dictionary
#     if not isinstance(cities, dict):
#         raise TypeError("'cities' must be a dictionary.")
    
#     # Update the cities dictionary with any new cities specified by the user
#     cities.update(new_cities)
    
#     # Compute the distances using vectorized operations
#     lats = df[lat_col]
#     lons = df[lon_col]
#     coords = list(zip(lats, lons))
    
#     unit_map = {
#         'km': 'kilometers',
#         'miles': 'miles',
#         'ft': 'feet',
#         'meters': 'meters',
#         'yards': 'yards',
#         'nautical': 'nautical'
#     }

#     def compute_distance(lat: float, lon: float, coord: Tuple[float, float], unit: str) -> float:
#         """
#         Compute the distance between a point and a city.

#         Parameters
#         ----------
#             lat : float
#                 The latitude of the point.
#             lon : float
#                 The longitude of the point.
#             coord : tuple of float
#                 The coordinates of the city as a tuple of (latitude, longitude).
#             unit : str
#                 The unit of distance to use. Acceptable values are 'km', 'miles', 'ft', 'meters', 'yards', and 'nautical'.

#         Returns
#         -------
#             distance : float
#                 The distance between the point and the city in the specified unit.

#         Raises
#         ------
#             ValueError
#                 If the unit is not recognized.
#         """
#         point = (lat, lon)
#         try:
#             return getattr(geodesic(point, coord), unit_map[unit])
#         except KeyError:
#             raise ValueError(f"Invalid unit: {unit}")
    
#     for city, coord in cities.items():
#         df[f'dist_{city}'] = df.apply(lambda x: compute_distance(x[lat_col], x[lon_col], coord, unit), axis=1)
    
#     df['dist_nearest_city'] = df[[f'dist_{city}' for city in cities]].min(axis=1)
    
#     return df

# # Define the cities and their coordinates
# cities = {
#     'Sac': (38.576931, -121.494949),
#     'SF':  (37.780080, -122.420160),
#     'SJ':  (37.334789, -121.888138),
#     'LA':  (34.052235, -118.243683),
#     'SD':  (32.715759, -117.163818)
# }

# # Create an instance of the DistanceParams dataclass
# params = DistanceParams(df_combined, 'Latitude', 'Longitude', cities)

# # Compute the distances and add them to df_combined
# df_combined = compute_distances(params)

# # Create a new instance of the DistanceParams dataclass for df_test
# params = DistanceParams(df_test, 'Latitude', 'Longitude', cities)

# # Compute the distances and add them to df_test
# df_test = compute_distances(params)

In [ ]:
# # Highlight newly created columns
# df_combined.iloc[:,-6:]

In [ ]:
# import json

# # Reads California coastline coordinates
# # (https://earthworks.stanford.edu/catalog/stanford-vx275xn8886) @kaivanbrunt
# def read_california_coastline(path):
#     """Read California coastline coordinates from a JSON file.
    
#     Parameters
#     ----------
#         path: str, path to the JSON file
        
#     Returns
#     -------
#         features: list of dicts, containing the features of the coastline
#     """
#     with open(path, 'r') as f:
#         coastline = json.load(f)
#         features = coastline['features']
    
#     return features

# # Unpacks California coastline coordinates and builds a dataframe
# def build_coastline_df(features):
#     """Build a Pandas DataFrame of California coastline coordinates.
    
#     Parameters
#     ----------
#         features: list of dicts, containing the features of the coastline
        
#     Returns
#     -------
#         coastline_df: Pandas DataFrame, containing the Longitude and Latitude of the coastline
#     """
#     coords = [feature['geometry']['coordinates'] for feature in features]
#     coords = np.hstack(coords).reshape((-1, 2))
#     coastline_df = pd.DataFrame(coords, columns=['Longitude', 'Latitude'])
    
#     return coastline_df

# # Finds the shortest distance to the coastline (Euclidian Distance)
# def get_shortest_distance_to_coastline(lat, lon, coastline_df):
#     """Calculate the shortest distance (Euclidean distance) from a given latitude and longitude to the coastline.
    
#     Parameters
#     ----------
#         lat: float, latitude
#         lon: float, longitude
#         coastline_df: Pandas DataFrame, containing the Longitude and Latitude of the coastline
        
#     Returns
#     -------
#         shortest_distance: float, shortest distance to the coastline
#     """
#     shortest_distance = (((coastline_df.Latitude - lat)**2 + (coastline_df.Longitude - lon)**2)**.5).min()
    
#     return shortest_distance

# # Read the California coastline coordinates
# path = r'/kaggle/input/coastline-of-california-coordinates/stanford-vx275xn8886-geojson.json'
# coastline_features = read_california_coastline(path)

# # Build a DataFrame of California coastline coordinates
# coastline_df = build_coastline_df(coastline_features)

# # Add a column to df_train and df_test with the shortest distance to the coastline
# def add_shortest_distance_to_coastline(df, coastline_df):
#     """Add a column to a Pandas DataFrame with the shortest distance to the coastline.
    
#     Parameters
#     ----------
#         df: Pandas DataFrame
#         coastline_df: Pandas DataFrame, containing the Longitude and Latitude of the coastline
        
#     Returns
#     -------
#         df: Pandas DataFrame, with an added column 'dist_to_cstl' containing the shortest distance to the coastline
#     """
#     df['dist_to_cstl'] = df.progress_apply(lambda row: get_shortest_distance_to_coastline(row.Latitude, row.Longitude, coastline_df), axis=1)
    
#     return df

# # Add shortest distance to coastline to df_train and df_test
# tqdm.pandas()
# df_combined = add_shortest_distance_to_coastline(df_combined, coastline_df)
# df_test  = add_shortest_distance_to_coastline(df_test, coastline_df)

In [ ]:
# # Highlight newly created columns
# df_combined[['dist_to_cstl']]

In [ ]:
def haversine_distance(lat: float, lng: float, degrees: bool = True) -> float:
    """
    Calculate the great circle distance between a point on Earth and the
    (0, 0) lat-long coordinate using the Haversine formula.
    
    Parameters
    ----------
        lat : float
            Latitude of the point.
        lng : float
            Longitude of the point.
        degrees : bool, optional
            If True, the input lat and lng values are in degrees. If False,
            they are in radians. Default is True.
        
    Returns
    -------
        distance : float
            Great circle distance between the point and the (0, 0) coordinate
            in kilometers.
    """
    r = 6371  # Earth's radius in kilometers
    
    # Convert decimal degrees to radians if necessary
    if degrees:
        lat, lng = map(np.radians, [lat, lng])
    
    # Haversine formula
    a = np.sin(lat / 2)**2 + np.cos(lat) * np.sin(lng / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    distance = r * c
    
    return distance

def manhattan_distance(lat: float, lng: float) -> float:
    """
    Calculate the Manhattan distance between a point and the (0, 0)
    lat-long coordinate.
    
    Parameters
    ----------
        lat : float
            Latitude of the point.
        lng : float
            Longitude of the point.
        
    Returns
    -------
        distance : float
            Manhattan distance between the point and the (0, 0) coordinate.
    """
    distance = np.abs(lat) + np.abs(lng)
    
    return distance

def euclidean_distance(lat: float, lng: float) -> float:
    """
    Calculate the Euclidean distance between a point and the (0, 0)
    lat-long coordinate.
    
    Parameters
    ----------
        lat : float
            Latitude of the point.
        lng : float
            Longitude of the point.
        
    Returns
    -------
        distance : float
            Euclidean distance between the point and the (0, 0) coordinate.
    """
    distance = (lat**2 + lng**2)**0.5
    
    return distance

def add_distance_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add distance metrics to a DataFrame based on latitude and longitude.
    
    Parameters
    ----------
        df : pd.DataFrame
            Input DataFrame containing 'Latitude' and 'Longitude' columns.
        
    Returns
    -------
        df : pd.DataFrame
            Output DataFrame with additional columns for the Haversine distance,
            Manhattan distance, and Euclidean distance between each point and
            the (0, 0) lat-long coordinate.
    """
    df['haversine'] = df[['Latitude', 'Longitude']].apply(
        lambda x: haversine_distance(x[0], x[1]), axis=1)
    df['manhattan'] = df[['Latitude', 'Longitude']].apply(
        lambda x: manhattan_distance(x[0], x[1]), axis=1)
    df['euclidean'] = df[['Latitude', 'Longitude']].apply(
        lambda x: euclidean_distance(x[0], x[1]), axis=1)
    
    return df

# Add the distance metrics to the DataFrames
df_combined = add_distance_metrics(df_combined)
df_test = add_distance_metrics(df_test)

In [ ]:
# Highlight newly created columns
df_combined[['haversine', 'manhattan', 'euclidean']]

In [ ]:
def add_custom_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add custom features to a DataFrame.
    
    Parameters
    ----------
        df : pd.DataFrame
            Input DataFrame.
        
    Returns
    -------
        df : pd.DataFrame
            Output DataFrame with additional columns.
    """
    df['DiagCoord']   = df['Longitude'] + df['Latitude']
    df['BedsPerRoom'] = df['AveBedrms'] / df['AveRooms']
#     df['OccupancyPerRoom']    = np.log1p(df['AveOccup'] / df['AveRooms'])
#     df['OccupancyPerBedRoom'] = np.log1p(df['AveOccup'] / df['AveBedrms'])
#     df['AvgRoomtoBedRoom']    = np.log1p(df['AveRooms'] / df['AveBedrms'])
#     df['OtherRooms']          = df['AveRooms'] - df['AveBedrms']
#     df['MedIncPop']           = np.log1p(df['MedInc'] / df['Population'])
#     df['RoomMedInc']          = np.log1p(df['AveRooms'] / df['MedInc'])
#     df['BedroomMedInc']       = np.log1p(df['AveBedrms'] / df['MedInc'])
#     df['AvgOccupMedInc']      = np.log1p(df['AveOccup'] / df['MedInc'])
    
    return df

# Add the custom features to the DataFrames
df_combined = add_custom_features(df_combined)
df_test = add_custom_features(df_test)

In [ ]:
# Highlight newly created columns
df_combined.iloc[:,-2:]

In [ ]:
features = df_combined.columns[df_combined.columns != TARGET]
print(f'Features: {features.tolist()}\n')

In [ ]:
X = df_combined[features]
y = df_combined[TARGET]

⬆️ [Back to the top](#Agenda) ⬆️
___
# <a id="Validation"><b>4 <span style='color:#a600ff'>•</span> Cross-validation method ✅</b></a>

In [ ]:
# Function modified from:
# https://scikit-learn.org/stable/auto_examples/model_selection/plot_cv_indices.html
# Inspired by https://www.kaggle.com/tomwarrens/timeseriessplit-how-to-use-it/notebook

from matplotlib.patches import Patch
from sklearn.model_selection import BaseCrossValidator

@dataclass
class PlotCVIndicesConfig:
    """
    Configuration data class for the `plot_cv_indices` function.
    
    Attributes
    ----------
        cv : Type[BaseCrossValidator]
            A scikit-learn cross-validation generator.
        X : np.ndarray
            The input array to be split into training and testing sets.
        y : np.ndarray
            The target array to be split into training and testing sets.
        n_splits : int
            The number of splits in the cross-validation process.
        date_col : Optional[pd.Series], optional
            A pandas Series containing dates corresponding to each sample in `X`. The dates will be displayed on the x-axis of the plot.
            The default is None.
        cmap : Optional[str], optional
            The name of the matplotlib colormap to use for the plot.
            The default is 'plasma'.
    """
    cv: Type[BaseCrossValidator]
    X: np.ndarray
    y: np.ndarray
    n_splits: int
    date_col: Optional[pd.Series] = None
    cmap: Optional[str] = 'plasma'

def plot_cv_indices(config: PlotCVIndicesConfig) -> plt.Axes:
    """
    Plots the indices of the training and testing sets for each iteration of a cross-validation process.
    
    Parameters
    ----------
        data : PlotCVIndicesData
            A data class containing the required parameters for the function.
    
    Returns
    -------
        ax : matplotlib Axes
            The Axes object containing the plot.
        
    Raises
    ------
        ValueError
            If the number of splits in `cv` does not match `n_splits`.
    """
    cv       = config.cv
    X        = config.X
    y        = config.y
    n_splits = config.n_splits
    date_col = config.date_col
    cmap     = config.cmap
    
    # Check that the number of splits in the cross-validation generator matches the provided value
    if hasattr(cv, 'get_n_splits'):
        n_splits_cv = cv.get_n_splits()
        if n_splits != n_splits_cv:
            raise ValueError(f'The number of splits in the cross-validation generator ({n_splits_cv}) does not match the provided value ({n_splits}).')
    else:
        n_splits_cv = n_splits
    
    # Create the figure and axis
    fig, ax = plt.subplots(figsize=(20, 15))
    
    # Iterate through the splits and plot the indices of the training and testing sets
    for ii, (tr, tt) in enumerate(cv.split(X=X, y=y)):
        indices = np.array([np.nan] * len(X))  # Initialize indices to nan
        indices[tt] = 1  # Set indices of the testing set to 1
        indices[tr] = 0  # Set indices of the training set to 0
        
        ax.scatter(
            range(len(indices)),
            [ii + 0.5] * len(indices),  # Set y-coordinate for each index
            c=indices,  # Use indices as colors
            marker="_",
            lw=20,
            cmap=plt.get_cmap(cmap),  # Use a color map
            vmin=-0.2,
            vmax=1.2,
            zorder=2
        )
    
    # Set y-tick labels
    yticklabels = [x+1 for x in list(range(n_splits_cv))]
    
    if date_col is not None:
        # Set x-tick locations and labels
        tick_locations  = ax.get_xticks()
        tick_dates = [" "] + date_col.iloc[list(tick_locations[1:-1])].astype(str).tolist() + [" "]
        tick_locations_str = [str(int(i)) for i in tick_locations]
        new_labels = ['\n\n'.join(x) for x in zip(list(tick_locations_str), tick_dates)]
        
        ax.set_xticks(tick_locations)
        ax.set_xticklabels(new_labels)
    
    # Set plot and axis properties
    ax.set_facecolor('#fcfcfc')
    ax.grid(alpha=0.7, linewidth=1, zorder=0)
    
    # Set y-tick locations and labels
    ax.set_yticks(np.arange(n_splits) + .5)
    ax.set_yticklabels(yticklabels)
    
    # Set axis labels and properties
    ax.set_xlabel('Sample index', fontsize=15, labelpad=10)
    ax.xaxis.set_tick_params(labelsize=12, pad=10, length=0)
    
    ax.set_ylabel('CV iteration', fontsize=15, labelpad=10)
    ax.set_ylim([n_splits+0.2, -.2])
    ax.yaxis.set_tick_params(labelsize=12, pad=10, length=0)
    
    
    # Add a legend
    ax.legend(
        [Patch(color=plt.get_cmap(cmap)(.12)), Patch(color=plt.get_cmap(cmap)(.8))],
        ['Training set', 'Validation set'],
        fontsize=12,
        loc='upper right',
        bbox_to_anchor=(1.005, 1.035),
        bbox_transform=ax.transAxes,
        ncol=2
    )
    
    # Add a title
    ax.set_title(
        f'{type(cv).__name__} (n_splits = {n_splits_cv})',
        fontsize=20,
        pad=10
    )
    
    # Remove the top and right spines and set the background color
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_facecolor('#f5f5f5')
    
    return ax

In [ ]:
from sklearn.model_selection import KFold

# Set the seed and number of splits for the KFold generator
SEED = 42
N_SPLITS = 10

# Initialize the KFold generator
folds = KFold(n_splits=N_SPLITS, random_state=SEED, shuffle=True)

# Configuration object for the plot_cv_indices function
cv_plot_config = PlotCVIndicesConfig(
    cv=folds,  # The KFold generator
    X=X,       # The input array
    y=y,       # The target array
    n_splits=folds.n_splits  # The number of splits in the cross-validation process
)

# Plot the cross-validation indices
plot_cv_indices(cv_plot_config);

⬆️ [Back to the top](#Agenda) ⬆️
___
# <a id="Training"><b>5 <span style='color:#a600ff'>•</span> Model Training and Inference 🏋️</b></a>

In [ ]:
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error

# Initialize lists for storing predictions and scores
y_pred, scores, fold_models = [], [], []

# Loop through the K folds
for fold, (train_id, valid_id) in enumerate(tqdm(folds.split(X, y), total=N_SPLITS)):
    
    # Split the data into training and validation sets
    X_train, X_valid = X.iloc[train_id], X.iloc[valid_id]
    y_train, y_valid = y.iloc[train_id], y.iloc[valid_id]
    
    # Convert the data to CatBoost-compatible Pool objects
    train_pool = Pool(X_train, y_train, cat_features=['place'])
    valid_pool = Pool(X_valid, y_valid, cat_features=['place'])

    # Define the model parameters
    params = {
        'iterations': 20000,
        'loss_function': 'RMSE',
    }
    
    # Create the model with the specified parameters
    model = CatBoostRegressor(**params)
    
    # Train the model on the training data
    model.fit(
        train_pool, 
        eval_set = valid_pool,
        early_stopping_rounds = 1000,
        verbose = False
    )
    
    # Evaluate the model on the validation set
    valid_pred = model.predict(valid_pool)
    valid_score = mean_squared_error(y_valid, valid_pred, squared=False)
    
    # Print the RMSE score for the current fold
    print(f'# \033[1;31;43m Fold {fold+1} \033[0;0m \t RMSE score: \033[1m{valid_score:6f}\033[0m')
    
    # Append the RMSE score to the scores list
    scores.append(valid_score)
    
    # Append the trained model to the fold_models list
    fold_models.append(model)
    
   # Make predictions on the test set and append them to the y_pred list
    y_pred.append(model.predict(df_test))

In [ ]:
print(f'Mean RMSE score: \033[1m{np.mean(scores):.6f}\033[0m')
print(f'Standard deviation: {np.std(scores):.6f}')

### <span style='color:#a600ff'>Feature importance</span>

In [ ]:
# Initialize an empty list to store the feature importances
feature_importances = []

# Loop through the models for each fold
for fold_model in fold_models:
    # Calculate the feature importances as a fraction of the total importances
    fi = fold_model.feature_importances_ / fold_model.feature_importances_.sum()
    # Append the importances to the list
    feature_importances.append(fi)
    
# Calculate the mean feature importances across all folds
mean_fi = np.mean(feature_importances, axis=0)

# Create a dataframe of the feature importances
df_feature_importance = (pd
 .DataFrame({
     'feature': X.columns,
     'importance': mean_fi
 })
 .sort_values(by='importance', ascending=False)
 .reset_index(drop=True)
)

# Set the figure size
fig, ax = plt.subplots(figsize=(12, 20))

# Create the bar plot
sns.barplot(x='importance', y='feature', data=df_feature_importance, orient='h', palette='plasma', edgecolor='black')

# Add a title and axis labels
plt.title('Feature importance', fontsize=16, pad=20)
plt.xlabel('Importance', fontsize=14, labelpad=10)
plt.ylabel('Features', fontsize=14, labelpad=10)

# Set the font size of the tick labels
ax.tick_params(axis='both', labelsize=14)

# Adjust the plot margins to make room for the rotated tick labels
plt.subplots_adjust(left=0.2, bottom=0.15)

# Add gridlines
plt.grid(b=True, which='both', color='#999999', linestyle='-', alpha=0.2)

# Show the plot
plt.show()

⬆️ [Back to the top](#Agenda) ⬆️
___
# <a id="Submit"><b>6 <span style='color:#a600ff'>•</span> Post-processing and Submission 📮</b></a>

### <span style='color:#a600ff'>Post-processing</span>

Remember that, from the EDA, we found that all houses with a price above 5 are given the value `5.00001`. Given this threshold effect for high value houses, I therefore choose to cap the predictions by replacing all those above 5 with this same value.

In [ ]:
# Replace values greater than 5 with 5.00001 for each array
y_pred = [np.where(arr > 5, target_max_val, arr) for arr in y_pred]

# Take the mean along the second dimension
final_pred = np.stack(y_pred).mean(axis=0)

In [ ]:
# Check if elements greater than 5 are equal to 5.00001
(final_pred[final_pred > 5] == target_max_val).all()

Due to floating point precision errors, the mean of values equal to `5.00001` becomes `5.0000100000000005`. To be honest, we could ignore it as the number has so many zeros and is close to the target value after all... But let's do it right!

In [ ]:
final_pred[final_pred > 5] = target_max_val

In [ ]:
# Second check
(final_pred[final_pred > 5] == target_max_val).all()

### <span style='color:#a600ff'>Submission</span>

In [ ]:
submission = pd.read_csv(data_dir / 'sample_submission.csv')

submission[TARGET] = final_pred

submission

In [ ]:
submission.to_csv('submission.csv', index=False)